# Tutorial 2: Writing Custom Features

EEGDash ships with a built-in feature bank, but the real power of the library is that you can add your own features with almost no boilerplate. A custom feature is just a plain Python/NumPy function with two lightweight decorators — no subclassing, no configuration files, no registration steps required.

By the end of this tutorial you will have implemented seven features from scratch, validated two of them against the built-in feature bank, and run the full pipeline on a mock dataset.

In [1]:
import numpy as np
from eegdash.features import (
    feature_predecessor,
    univariate_feature,
    FeatureExtractor,
    extract_features,
)

## 1. The Decorator System

Every EEGDash feature function needs two decorators stacked in a specific order:

- **`@feature_predecessor()`** — tells the pipeline what form the input data has. With no argument it means "I expect the raw signal". You can also pass a preprocessor function to say "I expect the output of *that* function".
- **`@univariate_feature`** — tells the pipeline that your function returns one scalar per channel, i.e. it collapses the time axis and produces an output of shape `(batch, channels)`.

**Decorator order matters:** `@feature_predecessor` must be the *outermost* decorator (listed first), and `@univariate_feature` must be listed second. The data shape contract is: input `x` has shape `(batch, channels, time)`, and a univariate feature should return `(batch, channels)` by reducing the last axis.

In [2]:
# Minimal "hello world" feature — computes the channel-wise mean
@feature_predecessor()   # outermost: declare input type (raw signal)
@univariate_feature      # innermost: one value per channel
def my_mean(x):
    """Mean amplitude per channel."""
    return x.mean(axis=-1)  # (batch, channels, time) → (batch, channels)

In [3]:
# Quick sanity check on a toy batch
x_toy = np.random.randn(2, 4, 256)  # 2 windows, 4 channels, 256 samples
out = my_mean(x_toy)
print(f"Input shape:  {x_toy.shape}")
print(f"Output shape: {out.shape}")  # should be (2, 4)

Input shape:  (2, 4, 256)
Output shape: (2, 4)


## 2. Implementing Signal Features

Now let's implement four practically useful features that all operate on the raw time-series signal. Each one collapses the time axis (`axis=-1`) and returns one value per channel. Pay attention to how `np.diff` and `np.sign` are used — the key is always operating on the last axis so the function works correctly regardless of the batch size or channel count.

In [4]:
# Feature 1: Zero crossings
# Counts how many times the signal changes sign — a rough measure of frequency content.
# np.signbit compares consecutive samples directly: True for negative, False for positive.
@feature_predecessor()
@univariate_feature
def my_zero_crossings(x):
    """Number of zero crossings per channel."""
    return np.sum(np.signbit(x[..., :-1]) != np.signbit(x[..., 1:]), axis=-1)

In [5]:
# Feature 2: Line length
# Mean absolute first difference — average step size between consecutive samples.
# High values indicate high-frequency or high-amplitude activity.
@feature_predecessor()
@univariate_feature
def my_line_length(x):
    """Mean absolute sample-to-sample difference per channel."""
    return np.abs(np.diff(x, axis=-1)).mean(axis=-1)

In [6]:
# Feature 3: Mean absolute deviation
# Average distance of each sample from the channel mean.
# More robust than variance because it uses absolute values instead of squares.
@feature_predecessor()
@univariate_feature
def my_mean_absolute_deviation(x):
    """Mean absolute deviation per channel."""
    return np.mean(np.abs(x - x.mean(axis=-1, keepdims=True)), axis=-1)

In [7]:
# Feature 4: Lag-1 autocorrelation
# Measures how correlated each sample is with the one before it.
# High positive values indicate a slow, smooth signal; values near zero indicate
# a fast or noisy signal. A useful proxy for the dominant frequency content.
@feature_predecessor()
@univariate_feature
def my_autocorrelation(x):
    """Lag-1 autocorrelation per channel."""
    x_centered = x - x.mean(axis=-1, keepdims=True)
    num = np.sum(x_centered[..., 1:] * x_centered[..., :-1], axis=-1)
    den = np.sum(x_centered ** 2, axis=-1)
    # Avoid division by zero for constant signals (den == 0 → return 0)
    out = np.zeros_like(num)
    mask = den != 0
    out[mask] = num[mask] / den[mask]
    return out

In [8]:
# Test all four on the same toy batch
x_toy = np.random.randn(2, 4, 256)

zc  = my_zero_crossings(x_toy)
ll  = my_line_length(x_toy)
mad = my_mean_absolute_deviation(x_toy)
ac  = my_autocorrelation(x_toy)

print(f"Zero crossings shape:         {zc.shape}")   # (2, 4)
print(f"Line length shape:            {ll.shape}")   # (2, 4)
print(f"Mean abs deviation shape:     {mad.shape}")  # (2, 4)
print(f"Autocorrelation shape:        {ac.shape}")   # (2, 4)

# Spot-check values look reasonable
print(f"\nZero crossings sample values: {zc[0]}")
print(f"Line length sample values:    {ll[0].round(2)}")
print(f"MAD sample values:            {mad[0].round(4)}")
print(f"Autocorrelation values:       {ac[0].round(4)}  # near 0 for white noise")

# Sanity check: a slow sine wave should have high positive autocorrelation
t = np.linspace(0, 1, 256, endpoint=False)
x_slow = np.tile(np.sin(2 * np.pi * 2 * t), (2, 4, 1))  # 2 Hz
print(f"\nSlow sine autocorr: {my_autocorrelation(x_slow)[0].round(4)}  # should be near +1")

Zero crossings shape:         (2, 4)
Line length shape:            (2, 4)
Mean abs deviation shape:     (2, 4)
Autocorrelation shape:        (2, 4)

Zero crossings sample values: [113 129 132 119]
Line length sample values:    [1.04 1.24 1.04 1.06]
MAD sample values:            [0.8207 0.8724 0.7266 0.7477]
Autocorrelation values:       [ 0.096  -0.0546 -0.0309 -0.0141]  # near 0 for white noise

Slow sine autocorr: [0.9988 0.9988 0.9988 0.9988]  # should be near +1


## 3. Validating Against the Feature Bank

EEGDash includes a built-in feature bank with vetted, optimised implementations of common features. Before using a custom implementation in production it is good practice to verify it produces exactly the same output as the reference. Here we test `my_zero_crossings` and `my_line_length` against their built-in equivalents on a controlled signal where the correct answer is easy to reason about.

In [9]:
from eegdash.features import signal_zero_crossings, signal_line_length

In [10]:
# Controlled signal: a sine wave at 10 Hz, sampled at 256 Hz, 1 second long.
# The 0.3-radian phase offset ensures no sample lands exactly on zero,
# so both implementations count sign changes identically.
# At 10 Hz we expect exactly 20 zero crossings per second (two per cycle).
sfreq = 256
t = np.linspace(0, 1, sfreq, endpoint=False)
sine = np.sin(2 * np.pi * 10 * t + 0.3)  # shape (256,)

# Broadcast to a batch of shape (2, 4, 256) — same signal in every window/channel
x_sine = np.tile(sine, (2, 4, 1))  # (2, 4, 256)
print(f"Test batch shape: {x_sine.shape}")
print(f"Min |value|: {np.abs(sine).min():.2e}  (well above zero)")

Test batch shape: (2, 4, 256)
Min |value|: 5.48e-03  (well above zero)


In [11]:
# Validate zero crossings
my_zc_out  = my_zero_crossings(x_sine)
ref_zc_out = signal_zero_crossings(x_sine)

np.testing.assert_array_almost_equal(my_zc_out, ref_zc_out)
print(f"Zero crossings match!  Values per channel: {my_zc_out[0]}")

Zero crossings match!  Values per channel: [20 20 20 20]


In [12]:
# Validate line length
my_ll_out  = my_line_length(x_sine)
ref_ll_out = signal_line_length(x_sine)

np.testing.assert_array_almost_equal(my_ll_out, ref_ll_out)
print(f"Line length match!  Values per channel: {my_ll_out[0].round(4)}")

Line length match!  Values per channel: [0.1555 0.1555 0.1555 0.1555]


Both assertions pass without error, which confirms our implementations are numerically identical to the feature bank. `my_mean_absolute_deviation`, `my_autocorrelation`, `my_spectral_mean_frequency`, and `my_alpha_theta_ratio` are genuinely new features not present in the built-in bank, so there is no reference to compare against — which is exactly why the decorator system exists: EEGDash makes it easy to add novel features.

## 4. Custom Preprocessor and Spectral Features

Some features are most naturally computed on a *transformed* version of the signal rather than the raw time-series. EEGDash handles this with the concept of a **preprocessor**: a function decorated with only `@feature_predecessor()` that takes the raw signal and returns an intermediate representation. Feature functions that depend on that preprocessor declare the dependency by passing it to `@feature_predecessor(my_preprocessor)`.

The preprocessor receives an extra keyword argument `_metadata` from the pipeline, which contains recording metadata such as the sampling frequency. This avoids hard-coding constants and makes the feature portable across datasets.

We build one preprocessor (`my_spectral_preprocessor`) and two features that consume it. The pipeline runs the preprocessor **exactly once** per batch regardless of how many features depend on it.

In [13]:
from scipy.signal import welch

# Preprocessor: raw signal → power spectral density via Welch's method
# Note: only @feature_predecessor() here — NOT @univariate_feature.
# The output is a tuple (frequencies, power) that downstream features consume.
@feature_predecessor()
def my_spectral_preprocessor(x, /, *, _metadata):
    """Compute Welch PSD for each window and channel.

    Returns
    -------
    f : ndarray, shape (n_freqs,)
    p : ndarray, shape (batch, channels, n_freqs)
    """
    sfreq = _metadata["info"]["sfreq"]
    f, p = welch(x, fs=sfreq, axis=-1)
    return f, p

In [14]:
# Feature: spectral mean frequency
# The weighted average frequency (centroid of the power spectrum).
# Depends on my_spectral_preprocessor — note the argument to @feature_predecessor.
@feature_predecessor(my_spectral_preprocessor)  # declare dependency
@univariate_feature
def my_spectral_mean_frequency(f, p, /):
    """Power-weighted mean frequency per channel."""
    return np.sum(f * p, axis=-1) / np.sum(p, axis=-1)

In [15]:
# Feature 6: Alpha-to-theta power ratio
# The ratio of alpha-band power (8–12 Hz) to theta-band power (4–8 Hz).
# In human EEG this ratio rises with relaxed wakefulness and drops with drowsiness
# or cognitive load — making it a widely used neuroscience biomarker.
# Both features share the same preprocessor, so the PSD is computed only once.
@feature_predecessor(my_spectral_preprocessor)  # reuses the same preprocessor
@univariate_feature
def my_alpha_theta_ratio(f, p, /):
    """Alpha (8–12 Hz) to theta (4–8 Hz) power ratio per channel."""
    theta_power = np.sum(p[..., (f >= 4) & (f < 8)],  axis=-1)
    alpha_power = np.sum(p[..., (f >= 8) & (f < 12)], axis=-1)
    out = np.zeros_like(alpha_power)
    mask = theta_power != 0
    out[mask] = alpha_power[mask] / theta_power[mask]
    return out

In [16]:
# Test the alpha/theta ratio
x_toy = np.random.randn(2, 4, 256).astype(np.float32)
f, p = my_spectral_preprocessor(x_toy, _metadata={"info": {"sfreq": 256}})
atr = my_alpha_theta_ratio(f, p)
print(f"Alpha/theta ratio shape: {atr.shape}")     # (2, 4)
print(f"Values (random signal):  {atr[0].round(3)}")

# Sanity check: inject a pure 10 Hz (alpha) tone → ratio should be very high
t = np.linspace(0, 1, 256, endpoint=False)
alpha_tone = np.tile(np.sin(2 * np.pi * 10 * t).astype(np.float32), (2, 4, 1))
f, p = my_spectral_preprocessor(alpha_tone, _metadata={"info": {"sfreq": 256}})
atr_alpha = my_alpha_theta_ratio(f, p)
print(f"Pure alpha signal ratio: {atr_alpha[0]}  # should be >> 1")

Alpha/theta ratio shape: (2, 4)
Values (random signal):  [0.161 0.822 2.393 1.086]
Pure alpha signal ratio: [2.2815575e+16 2.2815575e+16 2.2815575e+16 2.2815575e+16]  # should be >> 1


In [17]:
# Test the preprocessor directly
x_toy = np.random.randn(2, 4, 256).astype(np.float32)
f, p = my_spectral_preprocessor(x_toy, _metadata={"info": {"sfreq": 256}})
print(f"Frequency bins: {len(f)}")
print(f"Power shape:    {p.shape}")  # (2, 4, n_freqs)

Frequency bins: 129
Power shape:    (2, 4, 129)


In [18]:
# Test the spectral feature
mf = my_spectral_mean_frequency(f, p)
print(f"Mean frequency shape: {mf.shape}")  # (2, 4)
print(f"Mean frequency values (Hz): {mf[0].round(2)}")

Mean frequency shape: (2, 4)
Mean frequency values (Hz): [69.44 64.19 63.45 60.28]


## 5. Putting It All Together — FeatureExtractor

`FeatureExtractor` is the object that ties features into a reusable, composable pipeline. You can nest `FeatureExtractor` instances to group related features — for example, one group for raw-signal features and a separate group for spectral features. When a group needs a preprocessor, pass it as the `preprocessor` keyword argument. The pipeline takes care of running each preprocessor exactly once per batch, regardless of how many features depend on it.

In [19]:
# Build a nested extractor with all six custom features.
# The spectral group reuses my_spectral_preprocessor for both spectral features —
# the pipeline computes the PSD only once and fans it out to both consumers.
extractor = FeatureExtractor({
    "signal": FeatureExtractor({
        "zero_crossings": my_zero_crossings,
        "line_length":    my_line_length,
        "mad":            my_mean_absolute_deviation,
        "autocorr":       my_autocorrelation,
    }),
    "spectral": FeatureExtractor(
        {
            "mean_freq":        my_spectral_mean_frequency,
            "alpha_theta_ratio": my_alpha_theta_ratio,
        },
        preprocessor=my_spectral_preprocessor,
    ),
})

In [20]:
# Build a mock dataset using braindecode utilities.
# create_from_X_y treats each row of X as an independent single-window recording,
# so calling it with X.shape=(32, 8, 256) produces 32 EEGWindowsDataset objects,
# each containing exactly one window. Wrapping three such batches in a
# BaseConcatDataset gives 96 single-window elements total.
import pandas as pd
from braindecode.datasets import create_from_X_y, BaseConcatDataset

n_recordings = 3
datasets = []
for i in range(n_recordings):
    X = np.random.randn(32, 8, 256).astype(np.float32)  # 32 windows per batch
    y = np.zeros(32, dtype=np.int64)
    ds = create_from_X_y(
        X, y,
        drop_last_window=False,
        sfreq=256,
        ch_names=[f"ch{j}" for j in range(8)],
    )
    datasets.append(ds)

mock_ds = BaseConcatDataset(datasets)
print(f"Total single-window datasets: {len(mock_ds.datasets)}")  # 96
print(f"Total windows (len):          {len(mock_ds)}")           # 96
print(f"Type of one element:          {type(mock_ds.datasets[0]).__name__}")

Creating RawArray with float64 data, n_channels=8, n_times=256


    Range : 0 ... 255 =      0.000 ...     0.996 secs


Ready.


Creating RawArray with float64 data, n_channels=8, n_times=256


    Range : 0 ... 255 =      0.000 ...     0.996 secs


...

Ready.


Creating RawArray with float64 data, n_channels=8, n_times=256


    Range : 0 ... 255 =      0.000 ...     0.996 secs


Ready.


Total single-window datasets: 96
Total windows (len):          96
Type of one element:          EEGWindowsDataset


In [21]:
# Run the full feature extraction pipeline
features_ds = extract_features(mock_ds, extractor, batch_size=32, n_jobs=1)

# Each element of features_ds.datasets corresponds to one EEG window.
# Its .features DataFrame has 1 row (the window) and 48 columns (6 features × 8 channels).
first = features_ds.datasets[0]
print(f"Feature matrix shape: {first.features.shape}")   # (1, 48)
print(f"\nFeature columns ({len(first.features.columns)} total):")
for col in first.features.columns:
    print(f"  {col}")

Extracting features:   0%|          | 0/96 [00:00<?, ?it/s]

Extracting features: 100%|██████████| 96/96 [00:00<00:00, 1074.48it/s]

Feature matrix shape: (1, 48)

Feature columns (48 total):
  signal_zero_crossings_ch0
  signal_zero_crossings_ch1
  signal_zero_crossings_ch2
  signal_zero_crossings_ch3
  signal_zero_crossings_ch4
  signal_zero_crossings_ch5
  signal_zero_crossings_ch6
  signal_zero_crossings_ch7
  signal_line_length_ch0
  signal_line_length_ch1
  signal_line_length_ch2
  signal_line_length_ch3
  signal_line_length_ch4
  signal_line_length_ch5
  signal_line_length_ch6
  signal_line_length_ch7
  signal_mad_ch0
  signal_mad_ch1
  signal_mad_ch2
  signal_mad_ch3
  signal_mad_ch4
  signal_mad_ch5
  signal_mad_ch6
  signal_mad_ch7
  signal_autocorr_ch0
  signal_autocorr_ch1
  signal_autocorr_ch2
  signal_autocorr_ch3
  signal_autocorr_ch4
  signal_autocorr_ch5
  signal_autocorr_ch6
  signal_autocorr_ch7
  spectral_mean_freq_ch0
  spectral_mean_freq_ch1
  spectral_mean_freq_ch2
  spectral_mean_freq_ch3
  spectral_mean_freq_ch4
  spectral_mean_freq_ch5
  spectral_mean_freq_ch6
  spectral_mean_freq_ch7
  spe

Each element of `features_ds.datasets` corresponds to one EEG window. Because `create_from_X_y` wraps every row of `X` as a separate single-window dataset, each element's `.features` DataFrame has shape `(1, 48)`: 1 row for that window and 48 columns for 6 scalar features × 8 channels. To inspect all 96 windows at once you can concatenate with `pd.concat([d.features for d in features_ds.datasets])`.

## 6. Summary and Next Steps

Here is the complete decorator contract to remember:

| Goal | Decorators (outermost → innermost) |
|------|-------------------------------------|
| Univariate feature on raw signal | `@feature_predecessor()`, `@univariate_feature` |
| Univariate feature on preprocessor output | `@feature_predecessor(my_preprocessor)`, `@univariate_feature` |
| Preprocessor (raw → intermediate) | `@feature_predecessor()` only |

The six features built in this tutorial span two families:

| Feature | Family | In built-in bank? |
|---------|--------|-------------------|
| `my_zero_crossings` | Signal | Yes (used for validation) |
| `my_line_length` | Signal | Yes (used for validation) |
| `my_mean_absolute_deviation` | Signal | No |
| `my_autocorrelation` | Signal | No |
| `my_spectral_mean_frequency` | Spectral | No |
| `my_alpha_theta_ratio` | Spectral | No |

The two spectral features share a single `my_spectral_preprocessor` — the PSD is computed once and reused by both consumers, which is the key efficiency benefit of the preprocessor system.

**Tutorial 3** covers advanced usage: multivariate features (output shape `(batch, n_output_features)` instead of `(batch, channels)`), chaining preprocessors, and parallelising extraction across CPU cores with `n_jobs > 1`.